In [ ]:
import pandas as pd
import numpy as np
from collections import defaultdict
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, precision_score, recall_score, f1_score

data = pd.read_csv("POS_tagged.csv")

sentences = []
sentence = []
previous = None

for row in data.itertuples():
    if previous != row[1]:
        if sentence:
            sentences.append(sentence)
        sentence = []
        previous = row[1]
    sentence.append((row[2], row[3]))

if sentence:
    sentences.append(sentence)

train_data, test_data = train_test_split(sentences, test_size=0.2, random_state=42)

tags = set()
words = set()

for sent in train_data:
    for word, tag in sent:
        words.add(word)
        tags.add(tag)

tags = sorted(list(tags))
words = sorted(list(words))

initial = defaultdict(int)

for sent in train_data:
    initial[sent[0][1]] += 1

for tag in tags:
    initial[tag] = (initial[tag] + 1) / (len(train_data) + len(tags))

transition = defaultdict(lambda: defaultdict(int))

for sent in train_data:
    for i in range(len(sent) - 1):
        tag1 = sent[i][1]
        tag2 = sent[i + 1][1]
        transition[tag1][tag2] += 1

for tag1 in tags:
    total = sum(transition[tag1].values())
    for tag2 in tags:
        transition[tag1][tag2] = (transition[tag1][tag2] + 1) / (total + len(tags))

emission = defaultdict(lambda: defaultdict(int))
tag_count = defaultdict(int)

for sent in train_data:
    for word, tag in sent:
        emission[tag][word] += 1
        tag_count[tag] += 1

for tag in tags:
    for word in words:
        emission[tag][word] = (emission[tag][word] + 1) / (tag_count[tag] + len(words))

log_initial = {}
log_transition = defaultdict(dict)
log_emission = defaultdict(dict)

for tag in tags:
    log_initial[tag] = np.log(initial[tag])

for tag1 in tags:
    for tag2 in tags:
        log_transition[tag1][tag2] = np.log(transition[tag1][tag2])

for tag in tags:
    for word in words:
        log_emission[tag][word] = np.log(emission[tag][word])

def viterbi(sentence):
    T = len(sentence)
    N = len(tags)

    score = np.full((N, T), -np.inf)
    back = np.zeros((N, T), dtype=int)

    for i, tag in enumerate(tags):
        emit = log_emission[tag].get(sentence[0], np.log(1e-10))
        score[i, 0] = log_initial[tag] + emit

    for t in range(1, T):
        for j, tag in enumerate(tags):
            emit = log_emission[tag].get(sentence[t], np.log(1e-10))
            values = np.array([
                score[i, t - 1] + log_transition[prev][tag]
                for i, prev in enumerate(tags)
            ])
            back[j, t] = np.argmax(values)
            score[j, t] = np.max(values) + emit

    best_path = []
    last = np.argmax(score[:, T - 1])
    best_path.append(last)

    for t in range(T - 1, 0, -1):
        last = back[last, t]
        best_path.append(last)

    best_path.reverse()

    return [tags[i] for i in best_path]

actual = []
predicted = []

for sent in test_data:
    sentence_words = [w for w, t in sent]
    true_tags = [t for w, t in sent]
    pred_tags = viterbi(sentence_words)
    actual.extend(true_tags)
    predicted.extend(pred_tags)

print("Accuracy:", accuracy_score(actual, predicted))
print("Precision:", precision_score(actual, predicted, average="weighted", zero_division=0))
print("Recall:", recall_score(actual, predicted, average="weighted", zero_division=0))
print("F1 Score:", f1_score(actual, predicted, average="weighted", zero_division=0))
print(classification_report(actual, predicted, zero_division=0))

test_sentences = [
    "I love NLP",
    "The dog runs fast",
    "She is reading a book",
    "Artificial Intelligence is amazing",
    "They play football every day"
]

for s in test_sentences:
    w = s.split()
    print("\nSentence:", s)
    print(list(zip(w, viterbi(w))))